# Paper 3 — M2b: Full E1 Datasets

**Full E1 datasets** (spec v0.2): same pipeline as the M2 shakedown, scaled up for statistics —
20 UEs, 30 s, four regimes along the non-stationarity axis. Storage is reduced: full-resolution CSI
is kept only for each UE's serving array (the link E0/E1 filter), while per-array energies, clusters
and reports are kept for all 12 arrays (what E3 needs). ~250 MB per dataset instead of ~2.5 GB.

Run FOUR times, editing only the marked block in cell [3]:
| run | regime | n_scatterers | ue_speed | speed_label |
|---|---|---|---|---|
| 1 | static | 0 | (0.0, 0.15) | 'slow' |
| 2 | static | 0 | (0.8, 1.5) | 'walk' |
| 3 | dynamic | 10 | (0.8, 1.5) | 'walk' |
| 4 | dynamic | 30 | (0.8, 1.5) | 'walk' |

Chunks go to LOCAL disk (fast, and they would otherwise fill Drive); only the reduced HDF5 is saved
to Drive. A run takes ~20 min, so a lost session costs one run, not the campaign.

The original shakedown description follows.

**Milestone M2** (spec v0.2, section 4): the adopted **A12** geometry from M1, UEs walking on
outdoor-constrained random-waypoint trajectories, and mobile scatterers (vehicle- and person-sized
boxes) re-traced per snapshot. Output: CSI **time series** per (UE, array) — the input of the
UE-side adaptive filter — plus periodic measurement reports, in the same HDF5 conventions as M1.

Run this notebook **twice**: once with `regime='dynamic'` and once with `regime='static'`
(same UE trajectories, no scatterers — the paired E1 comparison). The quick-look cell [10]
compares the two runs when both exist.

M2 shakedown scale: 10 UEs, 20 s at 50 ms (400 snapshots) per regime, ~15–20 min on a T4.
The full E1 datasets (25 UEs, longer runs, N_s sweep) are generated in M4 with this same notebook.

In [ ]:
# %% [1] Environment setup — safe to re-run; auto-restarts ONCE after installing
import importlib.util, subprocess, sys

if importlib.util.find_spec('sionna') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sionna==2.0.1', 'h5py'], check=True)
    print('Install done — the runtime will now restart (EXPECTED).')
    print('When it comes back, run this cell again: it will skip the install.')
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)

!nvidia-smi -L
import sionna, sionna.rt
print('Sionna:', sionna.__version__, '| Sionna RT:', sionna.rt.__version__)

In [ ]:
# %% [2] Drive mount + canary
import os, uuid
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M2_DIR = os.path.join(BASE, 'm2')
CKPT_DIR = '/content/ckpt'                        # LOCAL: chunks are transient and would fill Drive
os.makedirs(CKPT_DIR, exist_ok=True)

def drive_canary(path=M2_DIR):
    token = str(uuid.uuid4())
    canary = os.path.join(path, '.canary')
    with open(canary, 'w') as f: f.write(token)
    with open(canary) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(canary)
    print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [3] Configuration
from dataclasses import dataclass
import numpy as np

@dataclass
class M2Config:
    # --- regime: EDIT between runs ---
    # ---- EDIT THIS BLOCK PER RUN ----
    regime: str = 'static'                         # 'dynamic' or 'static'
    n_scatterers: int = 0                          # 0 / 10 / 30
    ue_speed: tuple = (0.0, 0.15)                  # (0.0,0.15)='slow' quasi-static; (0.8,1.5)='walk'
    speed_label: str = 'slow'                      # 'slow' or 'walk' — goes into the file name
    # ---------------------------------
    # --- band (FR1, as M1) ---
    carrier_hz: float = 3.5e9
    bandwidth_hz: float = 50e6
    num_subcarriers: int = 64
    # --- geometry (A12 adopted at the M1 gate) ---
    area_center: tuple = (0.0, 50.0)
    area_half: float = 100.0
    ue_height: float = 1.5
    tx_height: float = 7.0
    n_outer: int = 8
    n_inner: int = 4
    tx_ring_radius: float = 80.0
    inner_ring_radius: float = 35.0
    tx_ring_jitter: float = 12.0
    # --- time ---
    snapshot_dt: float = 0.05                      # ray-tracing period (geometry update)
    fine_dt: float = 0.005                         # CSI sampling period, synthesized via per-path Doppler.
                                                   # 50 ms moves a pedestrian >0.5 lambda at 3.5 GHz -> phase
                                                   # fully decorrelated; 5 ms gives ~0.06 lambda, predictable.
    duration_s: float = 30.0                       # full E1 datasets
    # --- UEs ---
    n_ue: int = 20                                 # full E1 datasets
    # --- scatterers (half vehicles, half pedestrians) ---
    veh_dims: tuple = (4.5, 1.8, 1.5)
    ped_dims: tuple = (0.5, 0.5, 1.7)
    veh_speed: tuple = (3.0, 8.0)
    ped_speed: tuple = (0.5, 1.5)
    # --- ray tracing / reports (as M1) ---
    max_depth: int = 5
    gamma_db: float = -10.0
    rsrp_quant_db: float = 1.0
    toa_quant_s: float = 1.0 / 50e6
    max_neighbors: int = 8
    t_rep_s: float = 1.0                           # reporting period (spec: {0.2, 1, 5}; sweep in M4)
    seed: int = 2026

CFG = M2Config()
T_STEPS = int(round(CFG.duration_s / CFG.snapshot_dt))      # traced snapshots
N_SUB = int(round(CFG.snapshot_dt / CFG.fine_dt))           # Doppler substeps per traced snapshot
T_FINE = T_STEPS * N_SUB                                    # CSI samples
REP_EVERY = int(round(CFG.t_rep_s / CFG.fine_dt))
SC_SPACING = CFG.bandwidth_hz / CFG.num_subcarriers
N_S = CFG.n_scatterers if CFG.regime == 'dynamic' else 0
RUN_TAG = f'M2full_{CFG.regime}_{CFG.speed_label}_Ns{N_S}_U{CFG.n_ue}_T{T_STEPS}_seed{CFG.seed}'
lam = 3e8/CFG.carrier_hz
print(f'{RUN_TAG}: {T_STEPS} traces @ {CFG.snapshot_dt*1000:.0f} ms x {N_SUB} Doppler substeps '
      f'= {T_FINE} CSI samples @ {CFG.fine_dt*1000:.0f} ms')
print(f'wavelength {lam*100:.1f} cm | UE displacement per CSI sample: '
      f'{CFG.ue_speed[0]*CFG.fine_dt*100:.1f}-{CFG.ue_speed[1]*CFG.fine_dt*100:.1f} cm '
      f'({CFG.ue_speed[1]*CFG.fine_dt/lam:.2f} lambda) | reports every {REP_EVERY} samples')

In [ ]:
# %% [4] Scene + A12 arrays (regenerated with the M1 seed, weak TXs auto-relocated)
import mitsuba as mi
from sionna.rt import (load_scene, PlanarArray, Transmitter, Receiver,
                       PathSolver, subcarrier_frequencies, Camera)

scene = load_scene(sionna.rt.scene.munich, merge_shapes=True)
scene.frequency = CFG.carrier_hz
scene.tx_array = PlanarArray(num_rows=2, num_cols=4, vertical_spacing=0.5,
                             horizontal_spacing=0.5, pattern='tr38901', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=2, vertical_spacing=0.5,
                             horizontal_spacing=0.5, pattern='iso', polarization='V')

def ring(n, radius, offset, cfg, rng):
    ang = np.linspace(0, 2*np.pi, n + 1)[:-1] + offset + rng.uniform(-0.15, 0.15, n)
    r = radius + rng.uniform(-cfg.tx_ring_jitter, cfg.tx_ring_jitter, n)
    return np.stack([cfg.area_center[0] + r*np.cos(ang),
                     cfg.area_center[1] + r*np.sin(ang),
                     np.full(n, cfg.tx_height)], axis=1)

def tx_positions(cfg, rng):
    outer = ring(cfg.n_outer, cfg.tx_ring_radius, 0.0, cfg, rng)
    if cfg.n_inner == 0:
        return outer
    inner = ring(cfg.n_inner, cfg.inner_ring_radius, np.pi/cfg.n_inner, cfg, rng)
    return np.concatenate([outer, inner], axis=0)

TX_POS = tx_positions(CFG, np.random.default_rng(CFG.seed))   # same stream as M1 -> same A12 geometry
A = TX_POS.shape[0]
for a, p in enumerate(TX_POS):
    tx = Transmitter(name=f'tx_{a}', position=[float(v) for v in p])
    scene.add(tx)
    tx.look_at([CFG.area_center[0], CFG.area_center[1], CFG.ue_height])

FREQS = subcarrier_frequencies(CFG.num_subcarriers, SC_SPACING)
SOLVER = PathSolver()
center = [CFG.area_center[0], CFG.area_center[1], CFG.ue_height]

def probe_hits():
    names = []
    for i in range(12):
        ang = 2*np.pi*i/12
        p = [CFG.area_center[0] + 40*np.cos(ang), CFG.area_center[1] + 40*np.sin(ang), CFG.ue_height]
        scene.add(Receiver(name=f'probe_{i}', position=[float(v) for v in p])); names.append(f'probe_{i}')
    paths = SOLVER(scene, max_depth=CFG.max_depth)
    _, tau = paths.cir(normalize_delays=False, out_type='numpy')
    tau = np.asarray(tau); t = tau if tau.ndim == 3 else tau[:, 0, :, 0, :]
    for n in names: scene.remove(n)
    return (t >= 0).sum(axis=(0, 2)) if t.shape[1] == A else (t >= 0).sum(axis=(1, 2))

hits = probe_hits(); thr = 0.25*np.median(hits)
for a_idx in np.where(hits < thr)[0]:
    tx = scene.get(f'tx_{a_idx}')
    x0, y0 = TX_POS[a_idx, 0]-CFG.area_center[0], TX_POS[a_idx, 1]-CFG.area_center[1]
    ang0, r0 = np.arctan2(y0, x0), np.hypot(x0, y0)
    best_h, best_p = hits[a_idx], TX_POS[a_idx].copy()
    for dr in (-30, -20, -10, 10, 20):
        for dang in (-0.15, 0.0, 0.15):
            p = np.array([CFG.area_center[0]+(r0+dr)*np.cos(ang0+dang),
                          CFG.area_center[1]+(r0+dr)*np.sin(ang0+dang), CFG.tx_height])
            tx.position = mi.Point3f(*map(float, p)); tx.look_at(center)
            h = probe_hits()[a_idx]
            if h > best_h: best_h, best_p = h, p
    tx.position = mi.Point3f(*map(float, best_p)); tx.look_at(center)
    TX_POS[a_idx] = best_p
    print(f'tx_{a_idx} relocated: {hits[a_idx]} -> {best_h} probe paths at {np.round(best_p, 1)}')
print('Final probe hits:', probe_hits().tolist())
drive_canary(M2_DIR)
np.save(os.path.join(M2_DIR, f'tx_positions_A{A}.npy'), TX_POS)

In [ ]:
# %% [5] Outdoor-constrained random-waypoint UE trajectories (seeded, regime-independent)
def is_indoor(pos):
    o = mi.Point3f(mi.Float(pos[:, 0]), mi.Float(pos[:, 1]), mi.Float(pos[:, 2]))
    si = scene.mi_scene.ray_intersect(mi.Ray3f(o, mi.Vector3f(0, 0, 1)))
    return np.array(si.is_valid(), dtype=bool)

def sample_outdoor(n, rng):
    pts = []
    while len(pts) < n:
        c = rng.uniform(-CFG.area_half, CFG.area_half, size=(4*n, 2)) + np.array(CFG.area_center)
        c = np.concatenate([c, np.full((len(c), 1), CFG.ue_height)], 1).astype(np.float32)
        pts.extend(c[~is_indoor(c)])
    return np.array(pts[:n])

def segment_outdoor(p0, p1):
    n = max(2, int(np.linalg.norm(p1[:2]-p0[:2])))          # ~1 m sampling along the segment
    pts = np.linspace(p0, p1, n).astype(np.float32)
    return not is_indoor(pts).any()

def waypoint_walk(rng, T, dt):
    traj = np.zeros((T, 3), np.float32)
    pos = sample_outdoor(1, rng)[0]
    t = 0
    while t < T:
        for _ in range(50):
            target = sample_outdoor(1, rng)[0]
            if segment_outdoor(pos, target): break
        else:
            target = pos                                     # boxed in: stand still this leg
        speed = rng.uniform(*CFG.ue_speed)
        d = np.linalg.norm(target[:2]-pos[:2])
        steps = max(1, int(np.ceil(d/(speed*dt))))
        for s in range(steps):
            if t >= T: break
            pos = pos + (target-pos)/max(steps-s, 1)
            traj[t] = pos; t += 1
    return traj

UE_RNG = np.random.default_rng(CFG.seed + 2)   # dedicated stream: identical trajectories in BOTH regimes
UE_TRAJ = np.stack([waypoint_walk(UE_RNG, T_STEPS, CFG.snapshot_dt) for _ in range(CFG.n_ue)])
assert not is_indoor(UE_TRAJ.reshape(-1, 3)).any(), 'indoor point leaked into a trajectory'
np.save(os.path.join(M2_DIR, f'ue_traj_U{CFG.n_ue}_T{T_STEPS}_seed{CFG.seed}.npy'), UE_TRAJ)

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
for u in range(CFG.n_ue): plt.plot(UE_TRAJ[u, :, 0], UE_TRAJ[u, :, 1], lw=1)
plt.scatter(TX_POS[:, 0], TX_POS[:, 1], marker='^', s=80, c='k', label='arrays')
plt.axis('equal'); plt.legend(); plt.title('UE trajectories (outdoor-constrained)'); plt.show()
print('Trajectories:', UE_TRAJ.shape)

In [ ]:
# %% [6] Mobile scatterers (dynamic regime only): box meshes + ITU materials, outdoor linear motion
from sionna.rt import SceneObject, ITURadioMaterial

def write_box_ply(path, dx, dy, dz):
    """Axis-aligned box centered at the origin (so .position sets its center)."""
    hx, hy, hz = dx/2, dy/2, dz/2
    v = [(sx*hx, sy*hy, sz*hz) for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)]
    f = [(0,1,3),(0,3,2),(4,6,7),(4,7,5),(0,4,5),(0,5,1),
         (2,3,7),(2,7,6),(0,2,6),(0,6,4),(1,5,7),(1,7,3)]
    with open(path, 'w') as fh:
        fh.write('ply\nformat ascii 1.0\n')
        fh.write(f'element vertex {len(v)}\nproperty float x\nproperty float y\nproperty float z\n')
        fh.write(f'element face {len(f)}\nproperty list uchar int vertex_indices\nend_header\n')
        for x, y, z in v: fh.write(f'{x} {y} {z}\n')
        for tri in f: fh.write('3 ' + ' '.join(map(str, tri)) + '\n')

SCAT, SCAT_STATE = [], None
if N_S > 0:
    write_box_ply('/content/box_veh.ply', *CFG.veh_dims)
    write_box_ply('/content/box_ped.ply', *CFG.ped_dims)
    mat_metal = ITURadioMaterial('scat-metal', 'metal', thickness=0.01)        # vehicles
    mat_body = ITURadioMaterial('scat-body', 'concrete', thickness=0.1)        # person proxy (see note)
    S_RNG = np.random.default_rng(CFG.seed + 3)
    objs, state = [], []
    for i in range(N_S):
        is_veh = i < N_S//2
        obj = SceneObject(fname='/content/box_veh.ply' if is_veh else '/content/box_ped.ply',
                          name=f'scat_{i}', radio_material=mat_metal if is_veh else mat_body)
        objs.append(obj)
        p = sample_outdoor(1, S_RNG)[0]
        speed = S_RNG.uniform(*(CFG.veh_speed if is_veh else CFG.ped_speed))
        ang = S_RNG.uniform(0, 2*np.pi)
        z = (CFG.veh_dims[2] if is_veh else CFG.ped_dims[2])/2
        state.append([p[0], p[1], z, speed*np.cos(ang), speed*np.sin(ang)])
    scene.edit(add=objs)                                    # one call: reloads the scene once
    SCAT, SCAT_STATE = objs, np.array(state, np.float64)
    print(f'{N_S} scatterers added ({N_S//2} vehicles, {N_S-N_S//2} pedestrians)')
else:
    print('Static regime: no scatterers')

def step_scatterers(rng):
    """Linear motion; on hitting a building or the area edge, pick a new random direction."""
    if SCAT_STATE is None: return
    for i, obj in enumerate(SCAT):
        x, y, z, vx, vy = SCAT_STATE[i]
        for _ in range(20):
            nx, ny = x + vx*CFG.snapshot_dt, y + vy*CFG.snapshot_dt
            probe = np.array([[nx, ny, CFG.ue_height]], np.float32)
            inside_area = (abs(nx-CFG.area_center[0]) < CFG.area_half and
                           abs(ny-CFG.area_center[1]) < CFG.area_half)
            if inside_area and not is_indoor(probe)[0]:
                break
            ang = rng.uniform(0, 2*np.pi); v = np.hypot(vx, vy)
            vx, vy = v*np.cos(ang), v*np.sin(ang)
        else:
            nx, ny = x, y                                    # trapped: stay put this step
        SCAT_STATE[i] = [nx, ny, z, vx, vy]
        obj.position = mi.Point3f(float(nx), float(ny), float(z))

**Material note.** Vehicles are ITU metal. Pedestrians use ITU concrete as a rough proxy for the
human body (comparable reflection loss at FR1); a dedicated dielectric model is a refinement to
mention in the paper's limitations, not a blocker for E1, which needs *moving* scatterers more than
perfect ones.

In [ ]:
# %% [7] Snapshot loop: geometry update + per-path Doppler time evolution
# Each traced snapshot yields N_SUB CSI samples synthesized from path Doppler shifts, so the CSI
# series is phase-coherent without tracing 10x more often (spec section 7, mitigation (i)).
import glob, time

for u in range(CFG.n_ue):
    scene.add(Receiver(name=f'ue_{u}', position=[float(v) for v in UE_TRAJ[u, 0]]))

CHUNK = 20
RUN_CKPT = os.path.join(CKPT_DIR, RUN_TAG); os.makedirs(RUN_CKPT, exist_ok=True)
done = {int(f.split('_')[-1].split('.')[0]) for f in glob.glob(os.path.join(RUN_CKPT, 'chunk_*.npz'))}
MOTION_RNG = np.random.default_rng(CFG.seed + 4)
n_chunks = int(np.ceil(T_STEPS/CHUNK))
NT = scene.tx_array.array_size                     # elements per array (single-beam combining below)
print(f'[{RUN_TAG}] {T_STEPS} traces in {n_chunks} chunks | {len(done)} already done')

buf, t0 = [], time.time()
for t in range(T_STEPS):
    c = t // CHUNK
    if c in done:
        continue
    step_scatterers(MOTION_RNG)
    for u in range(CFG.n_ue):
        ue = scene.get(f'ue_{u}')
        ue.position = mi.Point3f(*map(float, UE_TRAJ[u, t]))
        nxt = UE_TRAJ[u, min(t+1, T_STEPS-1)]
        v = (nxt - UE_TRAJ[u, t])/CFG.snapshot_dt   # velocity drives the per-path Doppler
        ue.velocity = mi.Point3f(*map(float, v))
    if SCAT_STATE is not None:
        for i, obj in enumerate(SCAT):
            obj.velocity = mi.Point3f(float(SCAT_STATE[i, 3]), float(SCAT_STATE[i, 4]), 0.0)

    paths = SOLVER(scene, max_depth=CFG.max_depth)
    h = np.asarray(paths.cfr(frequencies=FREQS, sampling_frequency=1.0/CFG.fine_dt,
                             num_time_steps=N_SUB, normalize_delays=False, out_type='numpy'))
    # [U, rx_ant, A, tx_ant, N_SUB, S] -> single transmit beam per array (the UE cannot resolve
    # individual array elements) -> [N_SUB, U, rx_ant, A, S]
    h = h.sum(axis=3)/np.sqrt(NT)
    buf.append(np.moveaxis(h, 3, 0).astype(np.complex64))

    if len(buf) == CHUNK or t == T_STEPS-1:
        np.savez_compressed(os.path.join(RUN_CKPT, f'chunk_{c:04d}.npz'),
                            h=np.concatenate(buf), t0=c*CHUNK)
        el = time.time()-t0
        print(f'chunk {c+1}/{n_chunks} | trace {t+1}/{T_STEPS} | {el:5.1f} s elapsed '
              f'| ETA {el/(t+1)*(T_STEPS-t-1):5.0f} s')
        buf = []

In [ ]:
# %% [8] Assemble CSI series + derived quantities + periodic reports
files = sorted(glob.glob(os.path.join(RUN_CKPT, 'chunk_*.npz')))
H = np.concatenate([np.load(f)['h'] for f in files])         # [T_FINE, U, rx_ant, A, S]
assert H.shape[0] == T_FINE, f'incomplete run: {H.shape[0]}/{T_FINE} samples'
print('CSI series:', H.shape, '| expected [T_FINE, U, rx_ant, A, S]')

# Ground-truth positions on the fine grid (linear interpolation between traced snapshots)
tf = np.linspace(0, T_STEPS-1, T_FINE)
UE_FINE = np.stack([np.stack([np.interp(tf, np.arange(T_STEPS), UE_TRAJ[u, :, d])
                              for d in range(3)], -1) for u in range(CFG.n_ue)]).astype(np.float32)

E_lin = (np.abs(H)**2).sum(axis=(2, 4)) + 1e-30              # [T_FINE, U, A]
E_dB = 10*np.log10(E_lin)
members_gamma = E_dB >= E_dB.max(axis=2, keepdims=True) + CFG.gamma_db

# Storage reduction: keep full CSI only for each UE's dominant array over the run. Serving-array
# handover within a run is out of scope for E0/E1 (which filter one link) and is recorded below
# as the fraction of time the instantaneous strongest array differs from the stored one.
SERVE = E_lin.mean(0).argmax(1)                              # [U]
CSI_SERVE = np.stack([H[:, u, :, SERVE[u], :] for u in range(CFG.n_ue)], 1)   # [T_FINE, U, rx_ant, S]
handover = float((E_lin.argmax(2) != SERVE[None, :]).mean())
print(f'Stored serving arrays: {SERVE.tolist()} | strongest array differs from stored: {handover*100:.1f}% of samples')

rep_t = np.arange(0, T_FINE, REP_EVERY)
E_q = np.round(E_dB[rep_t]/CFG.rsrp_quant_db)*CFG.rsrp_quant_db
rank = np.argsort(-E_dB[rep_t], axis=2)
neighbor_mask = np.zeros_like(members_gamma[rep_t])
np.put_along_axis(neighbor_mask, rank[:, :, :CFG.max_neighbors], True, axis=2)
print(f'Reports at {len(rep_t)} instants | mean cluster size: {members_gamma.sum(2).mean():.2f}')

In [ ]:
# %% [9] HDF5 export (time-series layout)
import h5py, datetime

drive_canary(M2_DIR)
out = os.path.join(M2_DIR, f'paper3_{RUN_TAG}.h5')
assert not os.path.exists(out), f'{out} already exists — rename or delete it before overwriting'
with h5py.File(out, 'w') as f:
    f.create_dataset('csi', data=CSI_SERVE, compression='gzip')               # [T_FINE, U, rx_ant, S] serving link
    f.create_dataset('serving_array', data=SERVE.astype(np.int8))
    f.create_dataset('energy_dB', data=E_dB.astype(np.float32), compression='gzip')   # [T_FINE, U, A] all arrays
    f.create_dataset('positions', data=UE_FINE, compression='gzip')           # [U, T_FINE, 3] ground truth
    f.create_dataset('tx_positions', data=TX_POS.astype(np.float32))
    f.create_dataset('cluster_gt', data=members_gamma)                        # [T, U, A]
    rep = f.create_group('reports')
    rep.create_dataset('t_idx', data=rep_t)
    rep.create_dataset('E_dB_q', data=E_q.astype(np.float32))
    rep.create_dataset('neighbor_mask', data=neighbor_mask)
    f.attrs.update(dict(band='FR1', regime=CFG.regime, n_scatterers=N_S, scene='munich',
                        snapshot_dt=CFG.snapshot_dt, fine_dt=CFG.fine_dt, t_rep_s=CFG.t_rep_s,
                        carrier_hz=CFG.carrier_hz, bandwidth_hz=CFG.bandwidth_hz,
                        gamma_db=CFG.gamma_db, seed=CFG.seed, sionna=sionna.__version__,
                        speed_label=CFG.speed_label, ue_speed_min=CFG.ue_speed[0],
                        ue_speed_max=CFG.ue_speed[1], handover_frac=handover, csi_layout='serving_only',
                        created=datetime.datetime.now().isoformat()))
print('Saved:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')

In [ ]:
# %% [10] M2 quick-look: COHERENT temporal correlation, static vs dynamic
# M2 v1 plotted mean |rho| per snapshot pair, which averages magnitudes and hides phase rotation:
# it read ~0.93 while the naive-predictor NMSE was 2.79 (phase fully decorrelated). Here rho is
# averaged as a COMPLEX quantity, so it collapses exactly when the channel stops being predictable.
def link_metrics(h5path, max_lag=20):
    with h5py.File(h5path, 'r') as f:
        if 'fine_dt' not in f.attrs:
            return None                                      # pre-Doppler file: different sampling, never mix
        Hh = f['csi'][:]; dt = float(f.attrs['fine_dt'])
    serve = None                                             # csi is already the serving link
    rho_c, rho_e, nmse = np.zeros(max_lag+1), np.zeros(max_lag+1), []
    U = Hh.shape[1]
    for u in range(U):
        hv = Hh[:, u].reshape(Hh.shape[0], -1)
        nrm = np.linalg.norm(hv, axis=1) + 1e-30
        for L in range(max_lag+1):
            a = hv[:-L] if L else hv
            b = hv[L:]
            num = (a.conj()*b).sum(1)/(nrm[:-L if L else None]*nrm[L:])
            rho_c[L] += np.abs(num.mean())                   # coherent: complex average, then modulus
            rho_e[L] += np.abs(num).mean()                   # envelope: modulus, then average
        nmse.append((np.linalg.norm(hv[1:]-hv[:-1], axis=1)**2/(nrm[1:]**2)).mean())
    return rho_c/U, rho_e/U, float(np.mean(nmse)), dt

runs = {os.path.basename(p): p for p in glob.glob(os.path.join(M2_DIR, 'paper3_M2full_*.h5'))}
print('Runs found:', list(runs))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
res = {}
for name, p in runs.items():
    m = link_metrics(p)
    if m is None:
        print(f'SKIPPED (old pre-Doppler format, delete and re-run): {name}')
        continue
    rc, re_, nmse, dt = m
    res[name] = (rc, nmse)
    lag_ms = np.arange(len(rc))*dt*1000
    lab = name.replace('paper3_M2_', '').replace('.h5', '')
    ax[0].plot(lag_ms, rc, label=f'{lab} | naive NMSE {nmse:.3f}')
    ax[1].plot(lag_ms, re_, label=lab)
for a_, ttl in zip(ax, ('coherent |E[rho]|', 'envelope E[|rho|]')):
    a_.set_xlabel('lag [ms]'); a_.set_ylabel(ttl); a_.grid(alpha=0.3); a_.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(M2_DIR, 'm2_quicklook.png'), dpi=150); plt.show()

stat = [v for k, v in res.items() if '_static_' in k]
dyn = [v for k, v in res.items() if '_dynamic_' in k]
if stat and dyn:
    ratio = dyn[0][1]/max(stat[0][1], 1e-12)
    print()
    print(f'Naive-predictor NMSE: dynamic {dyn[0][1]:.4f} vs static {stat[0][1]:.4f} | ratio {ratio:.2f}')
    print('[CHECK 1] predictable channel (static NMSE < 0.5):', 'PASS' if stat[0][1] < 0.5 else 'FAIL')
    print('[CHECK 2] scatterers add innovation (ratio > 1.2):', 'PASS' if ratio > 1.2 else 'FAIL')
    print('(Both regimes share UE trajectories; the ratio isolates the scatterer contribution - the H2 signal.)')
else:
    print()
    print('Run the other regime to complete the paired comparison.')

In [ ]:
# %% [11] Zip backup to PC (end-of-session discipline)
import shutil
zip_path = shutil.make_archive(f'/content/paper3_m2_{CFG.regime}_backup', 'zip', M2_DIR)
from google.colab import files as colab_files
colab_files.download(zip_path)